In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings 
warnings.filterwarnings('ignore')

from scipy.stats import chi2, spearmanr
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

In [2]:
data = pd.read_csv('Dataset/Final Dataset.csv')
data.head()

,Unnamed: 0,EnergyRating,BerRating,CO2Rating,MainSpaceHeatingFuel,CountyName,DwellingTypeDescr,Year_of_Construction,NoStoreys,GroundFloorArea(sq m),...,VentilationMethod,DraftLobby,PercentageDraughtStripped,NoOfSidesSheltered,SuspendedWoodenFloor,InsulationType,InsulationThickness,WaterStorageVolume,SolarHotWaterHeating,SolarPVEnergyOutput
0,205462,C3,212.41,52.29,Heating Oil,Co. Cork,Semi-detached house,1972,2,110.360,...,Natural vent.,YES,100.0,2.0,No,Loose Jacket,30.0,157.0,NO,0.00
1,1090514,B3,135.62,39.65,Heating Oil,Co. Cork,Detached house,1973,2,297.680,...,Natural vent.,NO,100.0,3.0,No,Factory Insulated,25.0,162.0,NO,6462.00
2,1090515,B2,108.33,23.09,Mains Gas,Co. Dublin,Semi-detached house,1999,2,105.820,...,Natural vent.,NO,100.0,2.0,No,Factory Insulated,25.0,117.0,NO,2107.34
3,1090517,D1,225.34,53.23,Heating Oil,Co. Longford,Detached house,1986,1,76.320,...,Natural vent.,NO,100.0,1.0,No,Factory Insulated,35.0,117.0,NO,0.00
4,1090520,C1,153.36,36.86,Mains Gas,Co. Dublin,End of terrace house,1985,2,77.652,...,Natural vent.,NO,93.0,2.0,No,Factory Insulated,35.0,88.0,NO,3924.10


In [3]:
print(data.columns)
print(data.shape)
SEED = 25223721

Index(['Unnamed: 0', 'EnergyRating', 'BerRating', 'CO2Rating',
       'MainSpaceHeatingFuel', 'CountyName', 'DwellingTypeDescr',
       'Year_of_Construction', 'NoStoreys', 'GroundFloorArea(sq m)',
       'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor',
       'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea',
       'StructureType', 'PredominantRoofType', 'RoomInRoofArea',
       'LivingAreaPercent', 'MainWaterHeatingFuel', 'HSMainSystemEfficiency',
       'WHMainSystemEff', 'HSSupplHeatFraction', 'HeatSystemControlCat',
       'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues',
       'NoOfFansAndVents', 'VentilationMethod', 'DraftLobby',
       'PercentageDraughtStripped', 'NoOfSidesSheltered',
       'SuspendedWoodenFloor', 'InsulationType', 'InsulationThickness',
       'WaterStorageVolume', 'SolarHotWaterHeating', 'SolarPVEnergyOutput'],
      dtype='str')
(341935, 43)


Creating appropiate directories

In [4]:
dirs = ['Figures','Outputs','Models']
for d in dirs:
    os.makedirs(d,exist_ok = True)

In [5]:
data = data.drop(columns = ['Unnamed: 0'],errors = 'ignoore').reset_index(drop = True)
if os.path.exists('Dataser/train_idx.csv'):
    train_idx = pd.read_csv('Dataset/train_idx.csv').iloc[:,0].values
    test_idx = pd.read_csv('Dataset/ttest_idx.csv').iloc[:,0].values
else:
    data = data.dropna(subset = ['EnergyRating']) # Needed to ensuee the split happens correctly
    train_idx,test_idx = train_test_split(data.index.values,test_size = 0.2,random_state = SEED,stratify = data['EnergyRating'])
    pd.Series(train_idx).to_csv('Dataset/train_idx.csv',index = False)
    pd.Series(test_idx).to_csv('Dataset/test_idx.csv',index = False)

## PCA

In [6]:
# We keep only continuous values since PXCA only works on that
exclude = ['BerRating','CO2Rating','HeatSystemControlCat','HeatResponseResponseCat']
num_col = [c for c in data.select_dtypes("number").columns if c not in exclude and data[c].nunique () > 2]
x_train,x_test = data.loc[train_idx,num_col], data.loc[test_idx,num_col]
print("Numeric Columns Chossen: ",num_col)

Numeric Columns Chossen:  ['Year_of_Construction', 'NoStoreys', 'GroundFloorArea(sq m)', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'RoomInRoofArea', 'LivingAreaPercent', 'HSMainSystemEfficiency', 'WHMainSystemEff', 'HSSupplHeatFraction', 'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'InsulationThickness', 'WaterStorageVolume', 'SolarPVEnergyOutput']


Data Preperation (Training data only)

In [8]:
skew = x_train.skew()
skewed = [ c for c in num_col if skew[c] >1 and x_train[c].min() >=0]
print('Skewed Columns: ',skewed)

Skewed Columns:  ['GroundFloorArea(sq m)', 'UValueWall', 'UValueRoof', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'RoomInRoofArea', 'LivingAreaPercent', 'HSMainSystemEfficiency', 'WHMainSystemEff', 'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'WaterStorageVolume', 'SolarPVEnergyOutput']


In [12]:
# Applying log transformation to all the skewed columns
def log_transform(df):
    df = df.copy()
    df[skewed] = np.log1p(df[skewed])
    return df

prep = Pipeline([
    ("imp", SimpleImputer(strategy='median')), 
    ("sc", StandardScaler())
])
z_train = prep.fit_transform(log_transform(x_train))
z_test = prep.transform(log_transform(x_test))
assert z_train.shape[1] == len(num_col)


Correlation checking 

In [14]:
R = np.corrcoef(z_train, rowvar=False)
n, p = z_train.shape
logdet = np.log(np.clip(np.linalg.eigvalsh(R), 1e-12, None)).sum()   # robust when columns are near-duplicates
bart = -(n - 1 - (2 * p + 5) / 6) * logdet
print(f"Bartlett chi2 = {bart:,.0f}, p-value = {chi2.sf(bart, p * (p - 1) / 2):.4g}")

inv = np.linalg.pinv(R)
partial = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
off = ~np.eye(p, dtype=bool)
kmo = (R[off] ** 2).sum() / ((R[off] ** 2).sum() + (partial[off] ** 2).sum())
print(f"KMO = {kmo:.3f}  (above 0.6 means PCA is suitable)")

plt.figure(figsize=(9, 7.5))
plt.imshow(R, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(p), num_col, rotation=90, fontsize=7); plt.yticks(range(p), num_col, fontsize=7)
plt.colorbar(label="Correlation"); plt.title("Correlation matrix (prepared training data)")
plt.tight_layout(); plt.savefig("figures/pca_correlation.png", dpi=200); plt.close()

Bartlett chi2 = 2,752,091, p-value = 0
KMO = 0.869  (above 0.6 means PCA is suitable)


Fitting PCA 

In [ ]:
pca = PCA(random_stare = SEED).fit(z_train)
eig,evr = pca.explained_variance_,pca.explained_variance_ratio_
cum = evr.cumsum()
K_kaiser = int( (eig>1).sum() )
k_90 = int( np.argmax( cum >= 0.90 ) +1)
k_95 = int( np.argmax( cum >= 0.95 ) +1)

TypeError: PCA.__init__() got an unexpected keyword argument 'random_stare'. Did you mean 'random_state'?